# Qwen Text Eraser (image + mask)

This notebook sends input image + mask pairs to the remote eraser endpoint and saves:
- **Erased images** → `/home/ubuntu/Downloads/qwen_text/eraser_result`
- **Visualization triplets (input | mask | erased)** → `/home/ubuntu/Downloads/qwen_text/visualization`

You can run in **batch mode** (all images under `INPUT_ROOT`) or **single-pair mode**.

In [5]:
import base64
import io
from pathlib import Path
from typing import Optional, Union

import requests
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm

DEFAULT_GUIDANCE_SCALE = 15.0
DEFAULT_NUM_INFERENCE_STEPS = 28
DEFAULT_MASK_DILATION_RADIUS = 3
DEFAULT_MAX_SIDE_LENGTH = 1920

# Endpoint + auth (same as test copy.ipynb)
BASETEN_ENDPOINT = "https://model-yqvdk1eq.api.baseten.co/development/predict"
BASETEN_API_KEY = "ZMuoM428.x26ObfkmoAx2Lnn6VGcqqxOIRupKdYYu"

remote_client = requests.Session()
remote_client.headers.update(
    {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {BASETEN_API_KEY}",
    }
)

# Paths
INPUT_ROOT = Path("/home/ubuntu/Downloads/energyx-drive")
MASK_ROOT = Path("/home/ubuntu/Downloads/qwen_text/mask")
OUTPUT_ROOT = Path("/home/ubuntu/Downloads/qwen_text/eraser_result")
VIS_ROOT = Path("/home/ubuntu/Downloads/qwen_text/visualization")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
VIS_ROOT.mkdir(parents=True, exist_ok=True)

# Mode
RUN_BATCH = True
SINGLE_IMAGE_PATH = Path("/path/to/your/input.png")
SINGLE_MASK_PATH = Path("/path/to/your/mask.png")
SKIP_EXISTING = True

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}


In [6]:
def _iter_images(root: Path):
    if not root.exists():
        raise FileNotFoundError(f"Input root not found: {root}")
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            yield path


def _load_image_input(image_input: Union[str, Path, Image.Image], mode: str = "RGB") -> Image.Image:
    if isinstance(image_input, Image.Image):
        return image_input.convert(mode)
    path = Path(image_input)
    return Image.open(path).convert(mode)


def _load_mask_input(mask_input: Union[str, Path, Image.Image]) -> Image.Image:
    if mask_input is None:
        raise ValueError("mask_input must not be None")
    if isinstance(mask_input, Image.Image):
        return mask_input.convert("L")
    path = Path(mask_input)
    return Image.open(path).convert("L")


def _encode_image_to_base64(image_input: Union[str, Path, Image.Image], mode: str = "RGB") -> tuple[str, Image.Image]:
    image = _load_image_input(image_input, mode=mode)
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8"), image


def _encode_mask_to_base64(
    mask_input: Union[str, Path, Image.Image], *, match_size: Optional[Image.Image] = None
) -> str:
    mask = _load_mask_input(mask_input)
    if match_size is not None and mask.size != match_size.size:
        mask = mask.resize(match_size.size, Image.NEAREST)
    buffer = io.BytesIO()
    mask.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def _decode_image_from_base64(data: str, mode: str = "RGB") -> Image.Image:
    return Image.open(io.BytesIO(base64.b64decode(data))).convert(mode)


def build_payload(
    image_input: Union[str, Path, Image.Image],
    *,
    mask_input: Optional[Union[str, Path, Image.Image]] = None,
    guidance_scale: float = DEFAULT_GUIDANCE_SCALE,
    num_inference_steps: int = DEFAULT_NUM_INFERENCE_STEPS,
    mask_dilation_radius: int = DEFAULT_MASK_DILATION_RADIUS,
    max_side_length: int = DEFAULT_MAX_SIDE_LENGTH,
) -> dict[str, object]:
    image_b64, image_obj = _encode_image_to_base64(image_input)
    payload: dict[str, object] = {
        "image": image_b64,
        "guidance_scale": guidance_scale,
        "num_inference_steps": num_inference_steps,
        "mask_dilation_radius": mask_dilation_radius,
        "max_side_length": max_side_length,
    }
    if mask_input is not None:
        payload["mask"] = _encode_mask_to_base64(mask_input, match_size=image_obj)
    return payload


def invoke_remote(payload: dict[str, object], label: str = "remote", preview: bool = False) -> dict[str, object]:
    print(f"[info] Invoking {BASETEN_ENDPOINT} ({label})")
    response = remote_client.post(BASETEN_ENDPOINT, json=payload, timeout=180)
    response.raise_for_status()
    data: dict[str, object] = response.json()
    for key in ("output", "result", "prediction", "response"):
        nested = data.get(key) if isinstance(data, dict) else None
        if isinstance(nested, dict):
            data = nested
            break

    remote_image = _decode_image_from_base64(str(data["image_base64"]))
    remote_mask = _decode_image_from_base64(str(data["mask_base64"]), mode="L")

    if preview:
        print(f"[{label} image]")
        display(remote_image)
        print(f"[{label} mask]")
        display(remote_mask)

    data["_image_obj"] = remote_image
    data["_mask_obj"] = remote_mask
    return data


def save_visualization(input_image: Image.Image, mask: Image.Image, erased: Image.Image, out_path: Path) -> None:
    if input_image.size != erased.size:
        input_image = input_image.resize(erased.size, Image.LANCZOS)
    if mask.size != erased.size:
        mask = mask.resize(erased.size, Image.NEAREST)

    mask_rgb = mask.convert("RGB")
    canvas = Image.new("RGB", (erased.width * 3, erased.height), (0, 0, 0))
    canvas.paste(input_image, (0, 0))
    canvas.paste(mask_rgb, (erased.width, 0))
    canvas.paste(erased, (erased.width * 2, 0))
    canvas.save(out_path)


In [8]:
def process_pair(image_path: Path, mask_path: Path, rel: Optional[Path] = None) -> None:
    label = rel.as_posix() if rel else image_path.stem

    payload = build_payload(
        image_path,
        mask_input=mask_path,
        guidance_scale=DEFAULT_GUIDANCE_SCALE,
        num_inference_steps=DEFAULT_NUM_INFERENCE_STEPS,
        mask_dilation_radius=DEFAULT_MASK_DILATION_RADIUS,
        max_side_length=DEFAULT_MAX_SIDE_LENGTH,
    )
    result = invoke_remote(payload, label=label, preview=False)
    erased = result["_image_obj"]

    out_rel = rel if rel else image_path.name
    if isinstance(out_rel, Path):
        out_dir = OUTPUT_ROOT / out_rel.parent
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{image_path.stem}_erased.png"
    else:
        out_dir = OUTPUT_ROOT
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{image_path.stem}_erased.png"

    erased.save(out_path)

    vis_rel = rel if rel else image_path.name
    if isinstance(vis_rel, Path):
        vis_dir = VIS_ROOT / vis_rel.parent
        vis_dir.mkdir(parents=True, exist_ok=True)
        vis_path = vis_dir / f"{image_path.stem}_viz.png"
    else:
        vis_dir = VIS_ROOT
        vis_dir.mkdir(parents=True, exist_ok=True)
        vis_path = vis_dir / f"{image_path.stem}_viz.png"

    input_image = _load_image_input(image_path, mode="RGB")
    mask_image = _load_mask_input(mask_path)
    save_visualization(input_image, mask_image, erased, vis_path)


if RUN_BATCH:
    image_paths = sorted(_iter_images(INPUT_ROOT))
    if not image_paths:
        raise FileNotFoundError(f"No images found under: {INPUT_ROOT}")

    missing_masks = []
    skipped = 0

    for path in tqdm(image_paths, desc="erase"):
        rel = path.relative_to(INPUT_ROOT)
        mask_path = MASK_ROOT / rel.parent / f"{path.stem}_mask.png"
        if not mask_path.exists():
            missing_masks.append(rel.as_posix())
            continue

        out_path = OUTPUT_ROOT / rel.parent / f"{path.stem}_erased.png"
        if SKIP_EXISTING and out_path.exists():
            skipped += 1
            continue

        process_pair(path, mask_path, rel=rel)

    print(f"[done] erased images saved under: {OUTPUT_ROOT}")
    print(f"[done] visualizations saved under: {VIS_ROOT}")
    if skipped:
        print(f"[info] skipped existing: {skipped}")
    if missing_masks:
        print(f"[warn] missing masks: {len(missing_masks)} (showing up to 10)")
        print("  " + "\n  ".join(missing_masks[:10]))
else:
    if not SINGLE_IMAGE_PATH.exists():
        raise FileNotFoundError(f"Input image not found: {SINGLE_IMAGE_PATH}")
    if not SINGLE_MASK_PATH.exists():
        raise FileNotFoundError(f"Mask not found: {SINGLE_MASK_PATH}")
    process_pair(SINGLE_IMAGE_PATH, SINGLE_MASK_PATH)

erase:   0%|          | 0/41 [00:00<?, ?it/s]

[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (1-ALT2_EnergyX_Feed_053025_002.png)
[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (1-ALT_EnergyX_Feed_053025_002.png)
[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (1_EnergyX_Feed_053025_002.png)
[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (2-ALT2_EnergyX_Feed_053025_002.png)
[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (2-ALT3_EnergyX_Feed_053025_002.png)
[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (2-ALT4_EnergyX_Feed_053025_002.png)
[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (2-ALT_EnergyX_Feed_053025_002.png)
[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (2_EnergyX_Feed_053025_002.png)
[info] Invoking https://model-yqvdk1eq.api.baseten.co/development/predict (3-ALT2_EnergyX_Feed_053025_002.png)
[info] Invoki